In [32]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import matplotlib.pyplot as plt
from tqdm import tqdm  # for progress bar
import re


In [ ]:
# Initialize list to store book data
books_data = []

# Base URL for all 50 pages
base_url = "https://books.toscrape.com/catalogue/page-{}.html"

In [26]:
# Loop through all pages (1 to 50)
for page in tqdm(range(1, 51)):
    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all book containers
    books = soup.find_all('article', class_='product_pod')

    for book in books:
        title = book.h3.a['title']
        price = book.find('p', class_='price_color').text.strip()
        availability = book.find('p', class_='instock availability').text.strip()
        rating_class = book.p['class'][1]

        # Store as dictionary
        books_data.append({
            'Title': title,
            'Price': price,
            'Availability': availability,
            'Rating': rating_class
        })


100%|██████████| 50/50 [01:44<00:00,  2.09s/it]


In [ ]:
# Convert to DataFrame
df = pd.DataFrame(books_data)
print("Raw data collected successfully!")
print(df.head())

Raw data collected successfully!
                                   Title    Price Availability Rating
0                   A Light in the Attic  Â£51.77     In stock  Three
1                     Tipping the Velvet  Â£53.74     In stock    One
2                             Soumission  Â£50.10     In stock    One
3                          Sharp Objects  Â£47.82     In stock   Four
4  Sapiens: A Brief History of Humankind  Â£54.23     In stock   Five


## DATA CLEANING

In [ ]:
#(i) Ensure every value is treated as string first, then remove non-digit chars and convert to float
def clean_price(x):
    # Convert to string
    x_str = str(x)
    # Remove any character that is NOT a digit or dot
    x_clean = re.sub(r'[^\d.]', '', x_str)
    # Convert to float
    try:
        return float(x_clean)
    except:
        return 0.0  # fallback for empty or invalid strings

df['Price'] = df['Price'].apply(clean_price)

# Verify
print(df.head())
print(df.dtypes)


                                   Title  Price Availability Rating
0                   A Light in the Attic  51.77     In stock  Three
1                     Tipping the Velvet  53.74     In stock    One
2                             Soumission  50.10     In stock    One
3                          Sharp Objects  47.82     In stock   Four
4  Sapiens: A Brief History of Humankind  54.23     In stock   Five
Title            object
Price           float64
Availability     object
Rating           object
dtype: object


In [ ]:
# (ii) Standardize rating text to numeric values
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['Rating'] = df['Rating'].map(rating_map)

In [36]:
# (iii) Strip spaces from text columns
df['Title'] = df['Title'].str.strip()
df['Availability'] = df['Availability'].str.strip()

In [37]:
# (iv) Handle missing values
df.fillna({'Rating': 0}, inplace=True)

In [38]:
# Confirm structure
print("\nCleaned DataFrame info:")
print(df.info())


Cleaned DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Title         1000 non-null   object 
 1   Price         1000 non-null   float64
 2   Availability  1000 non-null   object 
 3   Rating        1000 non-null   int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 31.4+ KB
None
